# OSC variant vcf scoring

In [1]:
%%bash
module load miniconda3/24.1.2-py310
module load cuda/12.4.1
conda activate py311

In [71]:
from alphagenome_research.model import dna_model
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["TF_GPU_ALLOCATOR"]='cuda_malloc_async'

import matplotlib.pyplot as plt
import pandas as pd
import pysam
from pysam import VariantFile
from io import StringIO
from tqdm import tqdm
import os
# from dotenv import load_dotenv

pd.set_option('display.max_columns', None)


In [2]:
LMNA_START = 156_082_572
LMNA_END = 156_140_081
gene_symbol = "LMNA"
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)

BASE_PATH = '/users/PAS2905/coraalbers/'

AG_DATA_PATH = '/users/PAS2905/coraalbers/ag/ag_data/'

HG38_FASTA_PATH = '/users/PAS2905/coraalbers/ag/hg38.fa'
HG38_GTF_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf.gz.feather'
HG38_SPLICE_START_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_starts.feather'
HG38_SPLICE_END_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_ends.feather'

CLINVAR_PATH = '/users/PAS2905/coraalbers/ag/clinvar.vcf.gz'

HEART_UB = 'UBERON:0000948'
LV_UB = 'UBERON:0002084'

gtf = pd.read_feather(
    HG38_GTF_PATH
)

In [3]:
model = dna_model.create_from_huggingface( 
    'all_folds', 
    organism_settings={ 
        dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings( 
            fasta_path=HG38_FASTA_PATH, 
            
            gtf_feather_path=HG38_GTF_PATH, 
            splice_site_starts_feather_path=HG38_SPLICE_START_PATH, 
            splice_site_ends_feather_path=HG38_SPLICE_END_PATH, 
        ), dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings() } )

print('all folds model initialized!')

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

all folds model initialized!


ClinVar variants P/PL and B/LB extracted in clinvar_variant_extraction notebook

VCFs output from that notebook are input here

## convert vcf to csv

In [18]:
def get_vcf_names(vcf_path):
    """Finds the true column header line inside the VCF file."""
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#CHROM"):
                # Remove the leading '#' and split by tabs
                return line.strip("#").strip().split("\t")
    raise ValueError("No header line starting with '#CHROM' found.")


vcf_filename = f'{BASE_PATH}ag/variant-effects/osc/outputs/blb_with_nc.bed'
csv_filename = f'{BASE_PATH}ag/variant-effects/osc/outputs/blb_with_nc.csv'

column_names = ['CHROM', 'POS', 'ID', 'REF','ALT', 'QUAL', 'FILTER', 'INFO', 'NC_CHROM', 'NC_START', 'NC_END', 'BP_OVERLAP']

# Read the VCF file, skipping metadata rows starting with '##'
df = pd.read_csv(
    vcf_filename,
    comment="#",
    sep="\t",
    names=column_names,
    header=None,
    low_memory=False,
    index_col=False
)



In [19]:
df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,NC_CHROM,NC_START,NC_END,BP_OVERLAP
0,1,156114771,1245553,C,A,.,.,ALLELEID=1234223;CLNDISDB=MedGen:C3661900;CLND...,1,156081998,156114918,1
1,1,156114772,1291232,C,A,.,.,ALLELEID=1281051;CLNDISDB=MedGen:C3661900;CLND...,1,156081998,156114918,1
2,1,156114791,292829,T,C,.,.,AF_TGP=0.0008;ALLELEID=277441;CLNDISDB=.|MONDO...,1,156081998,156114918,1
3,1,156114821,1269019,G,C,.,.,ALLELEID=1261450;CLNDISDB=MedGen:C3661900;CLND...,1,156081998,156114918,1
4,1,156114831,292831,G,T,.,.,AF_TGP=0.02196;ALLELEID=277446;CLNDISDB=.|Huma...,1,156081998,156114918,1
...,...,...,...,...,...,...,...,...,...,...,...,...
194,1,156138478,1380476,C,A,.,.,ALLELEID=1421298;CLNDISDB=MedGen:CN230736|EFO:...,1,156137761,156138487,1
195,1,156138479,924728,C,T,.,.,AF_EXAC=0.0001;ALLELEID=915226;CLNDISDB=MONDO:...,1,156137761,156138487,1
196,1,156138480,918676,C,T,.,.,ALLELEID=915118;CLNDISDB=Human_Phenotype_Ontol...,1,156137761,156138487,1
197,1,156138484,4781680,C,G,.,.,"ALLELEID=4893206;CLNDISDB=MONDO:MONDO:0018993,...",1,156137761,156138487,1


In [20]:
df['CHROM'] = 'chr1'
df

# Export the clean structure to a CSV
df.to_csv(csv_filename, index=False)
print(f"Successfully exported genomic VCF data to {csv_filename}!")

Successfully exported genomic VCF data to /users/PAS2905/coraalbers/ag/variant-effects/osc/outputs/blb_with_nc.csv!


## predict variants
from https://www.alphagenomedocs.com/colabs/batch_variant_scoring.html 

In [61]:
csv_file = f'{BASE_PATH}ag/variant-effects/osc/outputs/plp_with_nc.csv'
vcf = pd.read_csv(csv_file, sep=',')
print(vcf.columns)
required_columns = ['ID', 'CHROM', 'POS', 'REF', 'ALT']
for column in required_columns:
  if column not in vcf.columns:
    raise ValueError(f'VCF file is missing required column: {column}.')

organism = 'human'  # @param ["human", "mouse"] {type:"string"}

# @markdown Specify length of sequence around variants to predict:
sequence_length = '1MB'  # @param ["16KB", "100KB", "500KB", "1MB"] { type:"string" }
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f'SEQUENCE_LENGTH_{sequence_length}'
]


# @markdown Specify which scorers to use to score your variants:
score_rna_seq = True  # @param { type: "boolean"}
score_cage = True  # @param { type: "boolean" }
score_procap = True  # @param { type: "boolean" }
score_atac = True  # @param { type: "boolean" }
score_dnase = True  # @param { type: "boolean" }
score_chip_histone = True  # @param { type: "boolean" }
score_chip_tf = True  # @param { type: "boolean" }
score_polyadenylation = False  # @param { type: "boolean" }
score_splice_sites = True  # @param { type: "boolean" }
score_splice_site_usage = True  # @param { type: "boolean" }
score_splice_junctions = True  # @param { type: "boolean" }



# Parse organism specification.
organism_map = {
    'human': dna_client.Organism.HOMO_SAPIENS,
    'mouse': dna_client.Organism.MUS_MUSCULUS,
}
organism = organism_map[organism]

# Parse scorer specification.
scorer_selections = {
    'rna_seq': score_rna_seq,
    'cage': score_cage,
    'procap': score_procap,
    'atac': score_atac,
    'dnase': score_dnase,
    'chip_histone': score_chip_histone,
    'chip_tf': score_chip_tf,
    'polyadenylation': score_polyadenylation,
    'splice_sites': score_splice_sites,
    'splice_site_usage': score_splice_site_usage,
    'splice_junctions': score_splice_junctions,
}

all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
# print(all_scorers)
selected_scorers = [
    all_scorers[key]
    for key in all_scorers
    if scorer_selections.get(key.lower(), False)
]



# Remove any scorers or output types that are not supported for the chosen organism.
unsupported_scorers = [
    scorer
    for scorer in selected_scorers
    if (
        organism.value
        not in variant_scorers.SUPPORTED_ORGANISMS[scorer.base_variant_scorer]
    )
    | (
        (scorer.requested_output == dna_client.OutputType.PROCAP)
        & (organism == dna_client.Organism.MUS_MUSCULUS)
    )
]
if len(unsupported_scorers) > 0:
  print(
      f'Excluding {unsupported_scorers} scorers as they are not supported for'
      f' {organism}.'
  )
  for unsupported_scorer in unsupported_scorers:
    selected_scorers.remove(unsupported_scorer)





Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO',
       'NC_CHROM', 'NC_START', 'NC_END', 'BP_OVERLAP'],
      dtype='object')


In [22]:
variant_scorers.RECOMMENDED_VARIANT_SCORERS.keys()


dict_keys(['ATAC', 'CONTACT_MAPS', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP', 'RNA_SEQ', 'RNA_SEQ_ACTIVE', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'POLYADENYLATION', 'ATAC_ACTIVE', 'DNASE_ACTIVE', 'CHIP_TF_ACTIVE', 'CHIP_HISTONE_ACTIVE', 'CAGE_ACTIVE', 'PROCAP_ACTIVE'])

In [23]:
scorer_titles = ['ATAC', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP', 'RNA_SEQ', 'RNA_SEQ_ACTIVE', 'ATAC_ACTIVE', 'DNASE_ACTIVE', 'CHIP_TF_ACTIVE', 'CHIP_HISTONE_ACTIVE', 'CAGE_ACTIVE', 'PROCAP_ACTIVE']



In [62]:
scorer_titles = ['SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS']

In [63]:
for k in range(len(scorer_titles)):
    scorer = scorer_titles[k]
    selected_scorers = [all_scorers[scorer]]

    # Score variants in the VCF file.
    results = []
    
    for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
        variant = genome.Variant(
          chromosome=str(vcf_row.CHROM),
          position=int(vcf_row.POS),
          reference_bases=vcf_row.REF,
          alternate_bases=vcf_row.ALT,
          name=vcf_row.ID,
        )
        
        interval = LMNA_INTERVAL.resize(sequence_length)
        
        variant_scores = model.score_variant(
          interval=interval,
          variant=variant,
          variant_scorers=selected_scorers,
          organism=organism,
        )
        results.append(variant_scores)
    
    df_scores = variant_scorers.tidy_scores(results)
    # df_scores = df_scores[(df_scores['ontology_curie'] == LV_UB)]
    
    
    # @markdown Other settings:
    download_predictions = True  # @param { type: "boolean" }
    
    if download_predictions:
      df_scores.to_csv(f'{BASE_PATH}ag/variant-effects/osc/outputs/blb_clinvar_scores/plp_with_nc.{scorer}.{sequence_length}.scores.csv', index=False)
    #   files.download('variant_scores.csv')
    
    print('completed')

100%|██████████| 66/66 [01:28<00:00,  1.35s/it]


completed


100%|██████████| 66/66 [02:44<00:00,  2.50s/it]


completed


100%|██████████| 66/66 [01:49<00:00,  1.65s/it]


completed


In [25]:
df_scores

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,ontology_curie,biosample_name,biosample_type,data_source,genetically_modified,raw_score


In [8]:
# Examine just the effects of the variants on specific ontology term
columns = [c for c in df_scores.columns if c != 'ontology_curie']
df_scores[(df_scores['ontology_curie'] == LV_UB)][columns]

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,raw_score
257,chr1:156114912:GCCGGCCATGGAGACC>G,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,-0.031775
562,chr1:156114913:CCGGCCATGGAGACCCCGTCCCAG>C,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,-0.027840
867,chr1:156115275:G>A,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,0.006259
1172,chr1:156115275:G>C,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,0.004074
1477,chr1:156115276:T>A,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,-0.006361
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18862,chr1:156137651:C>G,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,0.016888
19167,chr1:156137652:A>G,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,0.027676
19472,chr1:156137653:G>A,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,-0.013095
19777,chr1:156138486:A>G,chr1:155587039-156635615:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0002084 DNase-seq,.,DNase-seq,heart left ventricle,tissue,adult,encode,paired,False,0.037443


In [28]:
# lmna_var_scores = pd.read_csv(f'{BASE_PATH}ag/variant-effects/osc/outputs/lmna_variants_pathogenic_VUS_LMNA.524288.scores.csv')
# lmna_var_scores

active tracks
Biological Meaning: Instead of calculating how much a mutation changes a trait, it registers whether either of the alleles leaves that chromatin region actively accessible to transcriptional machinery.

Active Allele Scorers

In addition to the differential scores described above, we also provide scoring configurations that capture the absolute activity level associated with one of the alleles, rather than quantifying the change between REF and ALT. This is calculated by taking the maximum of the aggregated signals from the REF and ALT alleles over the masked central window or gene region.

We provide recommended active allele scorers for the following modalities:

    Gene expression (RNA-seq):
    across exons for a gene of interest.

    TSS activity (CAGE, PRO-cap):
    within a local 501-bp window centered at the variant.

    Chromatin Accessibility (ATAC-seq, DNase-seq):
    within a local 501-bp window centered at the variant.

    Transcription Factor binding (ChIP-TF):
    within a local 501-bp window centered at the variant.

    Histone modifications (ChIP-Histone):
    within a local 2001-bp window centered at the variant.


In [78]:
# get all csvs

# Define the target directory and the string to exclude
blb_dir = Path(f"{BASE_PATH}ag/variant-effects/osc/outputs/blb_clinvar_scores")
plp_dir = Path(f"{BASE_PATH}ag/variant-effects/osc/outputs/plp_clinvar_scores")

exclude_string = "ACTIVE"

# Get all files that DO NOT contain the exclude_string in their name
blb_files = [
    file.name for file in blb_dir.iterdir() 
    if file.is_file() and exclude_string not in file.name
]

print(blb_files)

plp_files = [
    file.name for file in plp_dir.iterdir() 
    if file.is_file() and exclude_string not in file.name
]
print(plp_files)

# blb_files = glob.glob(f"{BASE_PATH}ag/variant-effects/osc/outputs/blb_clinvar_scores/*.csv")
# plp_files = glob.glob(f"{BASE_PATH}ag/variant-effects/osc/outputs/plp_clinvar_scores/*.csv")



# read and combine csvs into one df
blb_df = pd.concat((pd.read_csv(BASE_PATH +'ag/variant-effects/osc/outputs/blb_clinvar_scores/'+ file) for file in blb_files), ignore_index=True)
blb_df = blb_df[(blb_df['ontology_curie'] == LV_UB)]
plp_df = pd.concat((pd.read_csv(BASE_PATH +'ag/variant-effects/osc/outputs/plp_clinvar_scores/'+ file) for file in plp_files), ignore_index=True)
plp_df = plp_df[(plp_df['ontology_curie'] == LV_UB)]


['blb_with_nc.ATAC.1048576.scores.csv', 'blb_with_nc.DNASE.1048576.scores.csv', 'blb_with_nc.CHIP_TF.1048576.scores.csv', 'blb_with_nc.CHIP_HISTONE.1048576.scores.csv', 'blb_with_nc.CAGE.1048576.scores.csv', 'blb_with_nc.PROCAP.1048576.scores.csv', 'blb_with_nc.RNA_SEQ.1048576.scores.csv', 'blb_with_nc.SPLICE_SITES.1048576.scores.csv', 'blb_with_nc.SPLICE_SITE_USAGE.1048576.scores.csv', 'blb_with_nc.SPLICE_JUNCTIONS.1048576.scores.csv']
['plp_with_nc.ATAC.1048576.scores.csv', 'plp_with_nc.CAGE.1048576.scores.csv', 'plp_with_nc.CHIP_TF.1048576.scores.csv', 'plp_with_nc.CHIP_HISTONE.1048576.scores.csv', 'plp_with_nc.PROCAP.1048576.scores.csv', 'plp_with_nc.RNA_SEQ.1048576.scores.csv', 'plp_with_nc.DNASE.1048576.scores.csv', 'plp_with_nc.SPLICE_SITE_USAGE.1048576.scores.csv', 'plp_with_nc.SPLICE_JUNCTIONS.1048576.scores.csv', 'plp_with_nc.SPLICE_SITES.1048576.scores.csv']


/tmp/ipykernel_1056661/3145029691.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  blb_df = pd.concat((pd.read_csv(BASE_PATH +'ag/variant-effects/osc/outputs/blb_clinvar_scores/'+ file) for file in blb_files), ignore_index=True)
/tmp/ipykernel_1056661/3145029691.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  plp_df = pd.concat((pd.read_csv(BASE_PATH +'ag/variant-effects/osc/outputs/plp_clinvar_scores/'+ file) for file in plp_files), ignore_index=True)


In [67]:
blb_df[(blb_df['ontology_curie'] == LV_UB)].output_type.unique()

array(['ATAC', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'RNA_SEQ',
       'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS'], dtype=object)

In [81]:
blb_df.head()

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,raw_score,transcription_factor,histone_mark,gtex_tissue
0,chr1:156114771:C>A,chr1:155587039-156635615:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",UBERON:0002084 ATAC-seq,.,ATAC-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,-0.016054,NaN,NaN,NaN
1,chr1:156114772:C>A,chr1:155587039-156635615:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",UBERON:0002084 ATAC-seq,.,ATAC-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,-0.011096,NaN,NaN,NaN
2,chr1:156114791:T>C,chr1:155587039-156635615:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",UBERON:0002084 ATAC-seq,.,ATAC-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,-0.035760,NaN,NaN,NaN
3,chr1:156114821:G>C,chr1:155587039-156635615:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",UBERON:0002084 ATAC-seq,.,ATAC-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,-0.007483,NaN,NaN,NaN
4,chr1:156114831:G>T,chr1:155587039-156635615:.,NaN,NaN,NaN,NaN,NaN,NaN,ATAC,"CenterMaskScorer(requested_output=ATAC, width=...",UBERON:0002084 ATAC-seq,.,ATAC-seq,UBERON:0002084,heart left ventricle,tissue,adult,encode,paired,False,0.036189,NaN,NaN,NaN


In [79]:
print(len(blb_df))
blb_df = blb_df[blb_df.track_strand != '-']
print(len(blb_df))

print(len(plp_df))
plp_df = plp_df[plp_df.track_strand != '-']
print(len(plp_df))

32426
20287
9004
4978


In [80]:
blb_stats = blb_df.groupby('output_type')['raw_score'].agg(['min', 'max', 'mean', 'median'])
print('B/LB variant score stats')
print(blb_stats)
print()
plp_stats = plp_df.groupby('output_type')['raw_score'].agg(['min', 'max', 'mean', 'median'])
print('P/LP variant score stats')
print(plp_stats)

B/LB variant score stats
                        min       max      mean    median
output_type                                              
ATAC              -0.120129  0.160728 -0.002012 -0.004838
CAGE              -0.209591  0.144718 -0.006040 -0.001852
CHIP_HISTONE      -0.089673  0.100513 -0.001025 -0.000483
CHIP_TF           -0.188190  0.148818  0.000277 -0.001700
DNASE             -0.128312  0.247282 -0.002237 -0.005489
RNA_SEQ           -0.333640  0.301085  0.000858  0.000000
SPLICE_JUNCTIONS   0.000000  1.264648  0.036362  0.013794
SPLICE_SITE_USAGE  0.000122  0.109375  0.008219  0.003906

P/LP variant score stats
                        min        max      mean    median
output_type                                               
ATAC              -2.431791   0.334482 -0.035424  0.014836
CAGE              -1.924527   0.410768 -0.307937 -0.177860
CHIP_HISTONE      -0.758511   0.082995 -0.019972 -0.003047
CHIP_TF           -2.323652   0.122335 -0.009162  0.006683
DNASE          

In [83]:
# Returns the row as a Series
max_row = plp_df.loc[plp_df['raw_score'].idxmax()]
max_row

variant_id              chr1:156135441:CACAACCACAGAGAAGGGTCGCAGGATGTGG...
scored_interval                                chr1:155587039-156635615:.
gene_id                                                   ENSG00000160789
gene_name                                                            LMNA
gene_type                                                  protein_coding
gene_strand                                                             +
junction_Start                                                  156136436
junction_End                                                    156136920
output_type                                              SPLICE_JUNCTIONS
variant_scorer                                     SpliceJunctionScorer()
track_name                          junction_UBERON:0002084 total RNA-seq
track_strand                                                            .
Assay title                                                 total RNA-seq
ontology_curie                        

In [ ]:
# # Score variants in the VCF file.
# results = []

# for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
#   variant = genome.Variant(
#       chromosome=str(vcf_row.CHROM),
#       position=int(vcf_row.POS),
#       reference_bases=vcf_row.REF,
#       alternate_bases=vcf_row.ALT,
#       name=vcf_row.ID,
#   )
#   interval = variant.reference_interval.resize(sequence_length)

#   variant_scores = model.score_variant(
#       interval=interval,
#       variant=variant,
#       variant_scorers=selected_scorers,
#       organism=organism,
#   )
#   results.append(variant_scores)

# df_scores = variant_scorers.tidy_scores(results)


# # @markdown Other settings:
# download_predictions = True  # @param { type: "boolean" }

# if download_predictions:
#   df_scores.to_csv(f'{BASE_PATH}ag/variant-effects/osc/outputs/lmna_variants_pathogenic_VUS_LMNA.scores.csv', index=False)
# #   files.download('variant_scores.csv')

# df_scores

In [ ]:
# df_scores.to_pickle('lmna_pathogenic_variant_scores.pkl')
# # @markdown Other settings:
# download_predictions = True  # @param { type: "boolean" }

# if download_predictions:
#   df_scores.to_csv('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores.csv', index=False)
#   # files.download('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores.csv')

In [ ]:
columns = [c for c in df_scores.columns if c != 'ontology_curie']
heart_df_scores = df_scores[(df_scores['ontology_curie'] == 'UBERON:0000948')][columns] # heart uberon identifier
heart_df_scores.to_csv('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_variant_scores_heart_UBERON0000948.csv', index=False)

In [18]:
heart_df_scores.head()

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,track_name,track_strand,Assay title,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score,quantile_score
387,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,DNASE,"CenterMaskScorer(requested_output=DNASE, width...",UBERON:0000948 DNase-seq,.,DNase-seq,heart,tissue,embryonic,encode,paired,False,NaN,NaN,NaN,0.141282,0.963955
2883,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K27me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K27me3,NaN,-0.177345,-0.999621
2884,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me1,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me1,NaN,-0.000861,-0.134700
2885,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K4me3,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K4me3,NaN,0.088124,0.988467
2886,chr1:156114693:C>T,chr1:156049157-156180229:.,None,None,None,None,None,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,UBERON:0000948 Histone ChIP-seq H3K9ac,.,Histone ChIP-seq,heart,tissue,embryonic,encode,single,False,NaN,H3K9ac,NaN,0.101405,0.989576
